# Lab 3: Data Cleaning and Preparation (Customer Churn)
**Aim:** Clean and prepare customer data, handle missing values, duplicates, outliers, encode, and split into train/test.

In [2]:

import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. Create dataset with data-quality issues
X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    random_state=42
)

df = pd.DataFrame(X, columns=[
    'Monthly_Charges',
    'Tenure_Months',
    'Support_Calls',
    'Data_Usage'
])

df['Monthly_Charges'] = (df['Monthly_Charges'] * 20 + 70).round(2)
df['Tenure_Months'] = (df['Tenure_Months'] * 10 + 30).round().astype(int)
df['Support_Calls'] = np.abs(df['Support_Calls'] * 2 + 3).round().astype(int)
df['Data_Usage'] = np.abs(df['Data_Usage'] * 5 + 10).round(2)

df['Contract'] = np.where(
    df['Tenure_Months'] > 36,
    'Long Term',
    'Month-to-Month'
)

df['Churn'] = np.where(y == 1, 'Yes', 'No')

# 2. Add missing values and duplicates
df.loc[5, 'Monthly_Charges'] = np.nan
df.loc[15, 'Contract'] = ' month-to-month '
df.loc[25, 'Contract'] = 'LONG TERM'
df.loc[35, 'Support_Calls'] = 100

df = pd.concat([df, df.iloc[[10]]], ignore_index=True)

# 3. Display dataset
print("Dataset created successfully!")
print("Dataset Shape:", df.shape)
print(df.head())


Dataset created successfully!
Dataset Shape: (201, 6)
   Monthly_Charges  Tenure_Months  Support_Calls  Data_Usage        Contract  \
0            74.71             37              3       13.08       Long Term   
1            81.60             40              6        8.18       Long Term   
2            59.58             34              5       10.85  Month-to-Month   
3            43.87             49              0        4.45       Long Term   
4            63.64             25              2       15.15  Month-to-Month   

  Churn  
0   Yes  
1   Yes  
2   Yes  
3    No  
4    No  


In [3]:
# 2. Clean & Preprocess
print("Missing before:", df.isnull().sum()['Monthly_Charges'])
df['Monthly_Charges'] = df['Monthly_Charges'].fillna(df['Monthly_Charges'].median())
df = df.drop_duplicates().reset_index(drop=True)
df['Contract'] = df['Contract'].str.strip().str.title()
df['Support_Calls'] = df['Support_Calls'].clip(upper=20)
df['Yearly_Charges'] = (df['Monthly_Charges'] * 12).round(2)
print("Contract counts:\n", df['Contract'].value_counts())
df.head()

Missing before: 1
Contract counts:
 Contract
Month-To-Month    103
Long Term          97
Name: count, dtype: int64


,Monthly_Charges,Tenure_Months,Support_Calls,Data_Usage,Contract,Churn,Yearly_Charges
0,74.71,37,3,13.08,Long Term,Yes,896.52
1,81.60,40,6,8.18,Long Term,Yes,979.20
2,59.58,34,5,10.85,Month-To-Month,Yes,714.96
3,43.87,49,0,4.45,Long Term,No,526.44
4,63.64,25,2,15.15,Month-To-Month,No,763.68


In [4]:
# 3. Normalize & Train/Test Split
num_cols = ['Monthly_Charges', 'Tenure_Months', 'Support_Calls', 'Data_Usage', 'Yearly_Charges']
df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])

X_clean = pd.get_dummies(df.drop('Churn', axis=1), columns=['Contract'], drop_first=True)
y_clean = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean)
print("Training set shape:", X_train.shape)
print("Testing set shape :", X_test.shape)

df.to_csv('Cleaned_Telecom_Customer_Churn.csv', index=False)
print("Cleaned churn data exported.")

Training set shape: (160, 6)
Testing set shape : (40, 6)
Cleaned churn data exported.
